# AXUM — Ge'ez inscription restoration (QLoRA)

Fine-tunes **Qwen2.5-1.5B-Instruct** to fill `[MISSING]` tokens in damaged Ge'ez text.

**Before this notebook (on your laptop):**
1. `python scripts/generate_dataset.py`
2. `python scripts/export_restoration_colab.py` → uploads `exports/geez_restoration_colab.zip`

**Runtime:** GPU (T4 free tier is enough). ~30–90 min for 1 epoch on ~24k rows.

**After training:** download the merged folder and load in **LM Studio** (same API as `llm_restoration.py`).

This notebook does **not** train OCR or artefact classification — those are separate models.

In [ ]:
# Check GPU (Runtime → Change runtime type → T4 GPU)
import torch
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → GPU"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
%%capture
# Unsloth + TRL for efficient QLoRA (matches AXUM .cursorrules stack)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets

In [ ]:
import os
from pathlib import Path

DATA_DIR = Path("/content/restoration_data")
DATA_DIR.mkdir(exist_ok=True)

TRAIN_JSONL = DATA_DIR / "geez_restoration_train.jsonl"
VAL_JSONL   = DATA_DIR / "geez_restoration_val.jsonl"
EVAL_JSONL  = DATA_DIR / "geez_restoration_eval.jsonl"

if not TRAIN_JSONL.exists():
    from google.colab import files
    print("Upload exports/geez_restoration_colab.zip from your AXUM project")
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    !unzip -q -o {zip_name} -d {DATA_DIR}
    # zip may flatten or nest; fix common layouts
    for p in Path("/content").rglob("geez_restoration_train.jsonl"):
        TRAIN_JSONL = p
        DATA_DIR = p.parent
        VAL_JSONL = DATA_DIR / "geez_restoration_val.jsonl"
        EVAL_JSONL = DATA_DIR / "geez_restoration_eval.jsonl"
        break

assert TRAIN_JSONL.exists(), f"Missing train JSONL in {DATA_DIR}"
print("Train:", TRAIN_JSONL)
print("Val:  ", VAL_JSONL, "exists:", VAL_JSONL.exists())

In [ ]:
from datasets import load_dataset

train_ds = load_dataset("json", data_files=str(TRAIN_JSONL), split="train")
val_ds = load_dataset("json", data_files=str(VAL_JSONL), split="train") if VAL_JSONL.exists() else None

print("Train rows:", len(train_ds))
if val_ds:
    print("Val rows:  ", len(val_ds))
print("Sample roles:", [m["role"] for m in train_ds[0]["messages"]])

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def format_messages(examples):
    """Convert messages column to a single text field for SFT."""
    texts = []
    for convo in examples["messages"]:
        texts.append(
            tokenizer.apply_chat_template(
                convo,
                tokenize=False,
                add_generation_prompt=False,
            )
        )
    return {"text": texts}

train_fmt = train_ds.map(format_messages, batched=True, remove_columns=train_ds.column_names)
val_fmt = val_ds.map(format_messages, batched=True, remove_columns=val_ds.column_names) if val_ds else None

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

OUTPUT_DIR = "/content/geez_restoration_lora"

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_fmt,
    eval_dataset=val_fmt,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=50,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir=OUTPUT_DIR,
        report_to="none",
        evaluation_strategy="steps" if val_fmt else "no",
        eval_steps=200 if val_fmt else None,
        save_steps=200,
        save_total_limit=2,
    ),
)

trainer_stats = trainer.train()
print(trainer_stats)

In [ ]:
# Quick sanity check — same prompt shape as AXUM inference (few-shot in user msg at runtime)
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

test_damaged = "ሰ[MISSING]ም"
test_messages = [
    {"role": "system", "content": "You restore ancient Ge'ez inscriptions. Respond ONLY with JSON."},
    {"role": "user", "content": f"Damaged text:\n{test_damaged}\n\nPeriod: Aksumite\nLocation: Aksum\n\nRespond ONLY with JSON."},
]
inputs = tokenizer.apply_chat_template(
    test_messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.2)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
# Save LoRA adapter + optional merged 16-bit for LM Studio
LORA_DIR = "/content/geez_restoration_lora_adapter"
MERGED_DIR = "/content/geez_restoration_merged"

model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)

model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")
print("Saved adapter:", LORA_DIR)
print("Saved merged: ", MERGED_DIR)

## Download and use on the rover laptop

1. Zip `geez_restoration_merged` and download from Colab.
2. In **LM Studio** → Load model from folder (or convert to GGUF if you prefer).
3. Start local server on port 1234.
4. AXUM `GeezRestorationEngine` will auto-select the loaded Qwen model.

**Optional:** run 2–3 epochs if val loss still drops; watch for overfitting on synthetic HHD lines.

**OCR reminder:** retrain with `notebooks/colab_geez_ocr_train.ipynb` and `exports/geez_merged_colab.zip` — separate from this notebook.

In [ ]:
# Package download (run after save cells)
!zip -r /content/geez_restoration_merged.zip /content/geez_restoration_merged
from google.colab import files
files.download("/content/geez_restoration_merged.zip")